# 05_modeling.ipynb

4주차 B팀 모델링 코드입니다. 입력 파일은 3주차 결과물인 `modeling_dataset.csv`입니다. `monthly_merged.csv`를 다시 불러오지 않습니다.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [5]:

candidate_paths = [
    Path("../outputs/tables/modeling_result_dataset.csv"),
    Path("outputs/tables/modeling_result_dataset.csv"),
    Path("../data/processed/modeling_dataset.csv"),
    Path("data/processed/modeling_dataset.csv"),
    Path("../processed/modeling_dataset.csv"),
    Path("modeling_dataset.csv"),
]

input_path = None
for path in candidate_paths:
    if path.exists():
        input_path = path
        break

if input_path is None:
    raise FileNotFoundError("modeling_result_dataset.csv 또는 modeling_dataset.csv를 찾을 수 없습니다.")

print("사용 데이터:", input_path)

df = pd.read_csv(input_path, encoding="utf-8-sig")
df = df.sort_values(["gu", "contract_month"]).reset_index(drop=True)

print(df.shape)
df.head()

사용 데이터: ../outputs/tables/modeling_result_dataset.csv
(875, 55)


,sggCd,gu,contract_month,avg_sale_price,med_sale_price,avg_sale_price_per_m2,med_sale_price_per_m2,sale_count,avg_jeonse_deposit,med_jeonse_deposit,...,volume_drop_flag,high_gap_flag,rent_shift_flag,stagnation_score,stagnation_level,kmeans_cluster,logistic_regression_pred,logistic_regression_risk_proba,random_forest_pred,random_forest_risk_proba
0,11680,강남구,2023-06,228844.322344,219000.0,2663.421804,2751.590856,273,82748.708333,70000.0,...,0,1,0,0.20,1단계_활발관찰,4,1,0.998751,1,0.996667
1,11680,강남구,2023-07,231213.471545,214000.0,2623.794249,2696.871985,246,86057.539001,73000.0,...,1,1,0,0.45,3단계_안정정체,4,0,0.242471,0,0.053333
2,11680,강남구,2023-08,232705.516014,212000.0,2612.997473,2722.987222,281,85542.612191,73000.0,...,0,1,0,0.45,3단계_안정정체,4,1,0.621007,0,0.263333
3,11680,강남구,2023-09,213373.786408,200000.0,2538.890572,2544.302857,206,90637.708372,80000.0,...,1,1,0,0.45,3단계_안정정체,4,0,0.017629,0,0.000000
4,11680,강남구,2023-10,252927.244898,220000.0,2709.602322,2745.621846,147,90321.675314,80000.0,...,1,1,0,0.45,3단계_안정정체,4,1,0.995030,1,0.993333


In [6]:

base_required_cols = [
    "gu", "contract_month", "jeonse_rate", "gap_rate",
    "sale_growth_1m", "jeonse_growth_1m", "growth_gap_1m",
    "sale_volume_growth_1m", "rent_volume_growth_1m", "monthly_ratio",
    "total_risk_score", "risk_target", "warning_flag"
]

missing_cols = [col for col in base_required_cols if col not in df.columns]

has_resident_score = ("user_risk_score" in df.columns) or ("resident_risk_index" in df.columns)
has_investor_score = ("investor_risk_score" in df.columns) or ("investor_risk_index" in df.columns)

if not has_resident_score:
    missing_cols.append("user_risk_score 또는 resident_risk_index")

if not has_investor_score:
    missing_cols.append("investor_risk_score 또는 investor_risk_index")

print("누락 컬럼:", missing_cols)

if missing_cols:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_cols}")

누락 컬럼: []


In [7]:

if "user_risk_score" not in df.columns and "resident_risk_index" in df.columns:
    df["user_risk_score"] = df["resident_risk_index"]

if "investor_risk_score" not in df.columns and "investor_risk_index" in df.columns:
    df["investor_risk_score"] = df["investor_risk_index"]

if "resident_risk_index" not in df.columns:
    df["resident_risk_index"] = df["user_risk_score"]

if "investor_risk_index" not in df.columns:
    df["investor_risk_index"] = df["investor_risk_score"]

if "resident_risk_grade" not in df.columns:
    if "user_risk_grade" in df.columns:
        df["resident_risk_grade"] = df["user_risk_grade"]
    else:
        df["resident_risk_grade"] = None

if "investor_risk_grade" not in df.columns:
    df["investor_risk_grade"] = None

# 이미 계산된 월세화 변화량이 있으면 그대로 사용하고,
# 없을 때만 새로 계산합니다.
if "monthly_ratio_diff" not in df.columns:
    df["monthly_ratio_diff"] = df.groupby("gu")["monthly_ratio"].diff().fillna(0)

if "monthly_shift_abs" not in df.columns:
    df["monthly_shift_abs"] = df["monthly_ratio_diff"].abs()

df[["resident_risk_index", "investor_risk_index", "monthly_ratio_diff"]].head()

,resident_risk_index,investor_risk_index,monthly_ratio_diff
0,0.809714,0.741314,0.000000
1,0.564800,0.522743,-0.049337
2,0.597429,0.591657,0.003316
3,0.519257,0.497086,0.006747
4,0.781086,0.781371,-0.018962


In [8]:

gap_q75 = df["gap_rate"].quantile(0.75)
monthly_shift_q75 = df["monthly_shift_abs"].quantile(0.75)

df["sale_stagnation_flag"] = (df["sale_growth_1m"].abs() <= 0.01).astype(int)
df["jeonse_stagnation_flag"] = (df["jeonse_growth_1m"].abs() <= 0.01).astype(int)
df["volume_drop_flag"] = (df["sale_volume_growth_1m"] < 0).astype(int)
df["high_gap_flag"] = (df["gap_rate"] >= gap_q75).astype(int)
df["rent_shift_flag"] = (df["monthly_shift_abs"] >= monthly_shift_q75).astype(int)

df["stagnation_score"] = (
    0.25 * df["sale_stagnation_flag"] +
    0.20 * df["jeonse_stagnation_flag"] +
    0.25 * df["volume_drop_flag"] +
    0.20 * df["high_gap_flag"] +
    0.10 * df["rent_shift_flag"]
)

def stagnation_level(score):
    if score <= 0.20:
        return "1단계_활발관찰"
    elif score <= 0.40:
        return "2단계_완만정체"
    elif score <= 0.60:
        return "3단계_안정정체"
    elif score <= 0.80:
        return "4단계_신중정체"
    else:
        return "5단계_거래위축정체"

df["stagnation_level"] = df["stagnation_score"].apply(stagnation_level)

df["stagnation_level"].value_counts()

stagnation_level
1단계_활발관찰      354
2단계_완만정체      304
3단계_안정정체      176
4단계_신중정체       38
5단계_거래위축정체      3
Name: count, dtype: int64

In [9]:

features = [
    "jeonse_rate",
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "sale_volume_growth_1m",
    "rent_volume_growth_1m",
    "monthly_ratio",
]

target = "risk_target"

X = df[features].copy()
y = df[target].astype(int)

print(X.shape, y.shape)
print(y.value_counts())

(875, 8) (875,)
risk_target
0    656
1    219
Name: count, dtype: int64


In [10]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

df["kmeans_cluster"].value_counts().sort_index()

kmeans_cluster
0    179
1    108
2    284
3    107
4    197
Name: count, dtype: int64

In [11]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
}

result_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    result_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
    })
    
    pred_col = name.lower().replace(" ", "_") + "_pred"
    prob_col = name.lower().replace(" ", "_") + "_risk_proba"
    df[pred_col] = model.predict(X)
    if hasattr(model, "predict_proba"):
        df[prob_col] = model.predict_proba(X)[:, 1]

model_comparison = pd.DataFrame(result_rows)
model_comparison

,model,accuracy,precision,recall,f1_score
0,Logistic Regression,0.965714,0.895833,0.977273,0.934783
1,Random Forest,0.960000,0.951220,0.886364,0.917647


In [12]:

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": random_forest_model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

feature_importance

,feature,importance
0,sale_growth_1m,0.328288
1,growth_gap_1m,0.186424
2,jeonse_rate,0.152747
3,gap_rate,0.149712
4,monthly_ratio,0.059252
5,sale_volume_growth_1m,0.051125
6,jeonse_growth_1m,0.041714
7,rent_volume_growth_1m,0.030739


In [13]:

cluster_summary = df.groupby("kmeans_cluster").agg(
    sample_count=("gu", "count"),
    avg_total_risk_score=("total_risk_score", "mean"),
    avg_resident_risk_index=("resident_risk_index", "mean"),
    avg_investor_risk_index=("investor_risk_index", "mean"),
    avg_stagnation_score=("stagnation_score", "mean"),
    avg_gap_rate=("gap_rate", "mean"),
    avg_sale_growth_1m=("sale_growth_1m", "mean"),
    avg_growth_gap_1m=("growth_gap_1m", "mean"),
    avg_monthly_ratio=("monthly_ratio", "mean"),
    risk_target_rate=("risk_target", "mean"),
    warning_rate=("warning_flag", "mean")
).reset_index()

cluster_summary["risk_rank"] = cluster_summary["avg_total_risk_score"].rank(method="first").astype(int)
rank_to_label = {
    1: "안정권",
    2: "양호",
    3: "관찰",
    4: "신중검토",
    5: "우선점검"
}
cluster_summary["cluster_risk_label"] = cluster_summary["risk_rank"].map(rank_to_label)
cluster_summary = cluster_summary.sort_values("avg_total_risk_score", ascending=False).reset_index(drop=True)
cluster_summary

,kmeans_cluster,sample_count,avg_total_risk_score,avg_resident_risk_index,avg_investor_risk_index,avg_stagnation_score,avg_gap_rate,avg_sale_growth_1m,avg_growth_gap_1m,avg_monthly_ratio,risk_target_rate,warning_rate,risk_rank,cluster_risk_label
0,3,107,0.742676,0.717719,0.767633,0.238318,0.541794,0.081749,0.117370,0.447966,0.925234,0.570093,5,우선점검
1,4,197,0.576213,0.570879,0.581548,0.395685,0.566425,-0.004747,-0.018549,0.436807,0.365482,0.126904,4,신중검토
2,2,284,0.474066,0.471353,0.476778,0.214613,0.448633,0.015454,0.014006,0.392471,0.105634,0.007042,3,관찰
3,0,179,0.451424,0.467058,0.435791,0.210615,0.393081,0.008595,0.007271,0.528368,0.100559,0.000000,2,양호
4,1,108,0.273889,0.289569,0.258210,0.269907,0.394377,-0.069759,-0.133203,0.452403,0.000000,0.000000,1,안정권


In [14]:

latest_month = df["contract_month"].max()
latest_region_risk_ranking = df[df["contract_month"] == latest_month].copy()

ranking_cols = [
    "gu", "contract_month", "total_risk_score", "resident_risk_index", "investor_risk_index",
    "risk_grade", "resident_risk_grade", "investor_risk_grade", "warning_flag", "kmeans_cluster",
    "stagnation_score", "stagnation_level", "gap_rate", "sale_growth_1m", "growth_gap_1m", "monthly_ratio"
]
ranking_cols = [c for c in ranking_cols if c in latest_region_risk_ranking.columns]

latest_region_risk_ranking = latest_region_risk_ranking[ranking_cols].sort_values(
    "total_risk_score", ascending=False
).reset_index(drop=True)
latest_region_risk_ranking.insert(0, "rank", range(1, len(latest_region_risk_ranking) + 1))

latest_region_risk_ranking.head(10)

,rank,gu,contract_month,total_risk_score,resident_risk_index,investor_risk_index,risk_grade,resident_risk_grade,investor_risk_grade,warning_flag,kmeans_cluster,stagnation_score,stagnation_level,gap_rate,sale_growth_1m,growth_gap_1m,monthly_ratio
0,1,송파구,2026-04,0.896371,0.912171,0.880571,높음,높음,높음,1,4,0.20,1단계_활발관찰,0.640200,0.120449,0.085179,0.498288
1,2,강동구,2026-04,0.803400,0.809086,0.797714,높음,높음,높음,1,3,0.45,3단계_안정정체,0.569216,0.094946,0.121098,0.483731
2,3,서초구,2026-04,0.778343,0.778571,0.778114,높음,높음,높음,1,4,0.65,4단계_신중정체,0.548908,0.073769,0.072443,0.464730
3,4,강남구,2026-04,0.766686,0.786286,0.747086,높음,높음,높음,1,4,0.20,1단계_활발관찰,0.631588,0.042768,-0.004739,0.512939
4,5,성동구,2026-04,0.719743,0.685657,0.753829,높음,높음,높음,0,4,0.20,1단계_활발관찰,0.571284,0.037394,0.070984,0.420099
5,6,양천구,2026-04,0.670086,0.678000,0.662171,높음,높음,높음,0,2,0.20,1단계_활발관찰,0.518376,0.045872,0.039516,0.425320
6,7,구로구,2026-04,0.665971,0.639029,0.692914,높음,주의,높음,0,3,0.60,3단계_안정정체,0.525067,-0.002355,0.144124,0.492773
7,8,마포구,2026-04,0.660529,0.673514,0.647543,높음,높음,주의,0,4,0.35,2단계_완만정체,0.503796,0.027996,0.010762,0.531250
8,9,관악구,2026-04,0.611571,0.647771,0.575371,주의,높음,주의,0,2,0.00,1단계_활발관찰,0.440829,0.061471,0.044592,0.470405
9,10,금천구,2026-04,0.609829,0.666971,0.552686,주의,높음,주의,0,0,0.10,1단계_활발관찰,0.399560,0.086777,0.058286,0.559006


In [15]:

out_dir = Path("../outputs/tables/")
out_dir.mkdir(parents=True, exist_ok=True)

model_comparison.to_csv(out_dir / "model_comparison.csv", index=False, encoding="utf-8-sig")
feature_importance.to_csv(out_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(out_dir / "kmeans_cluster_summary.csv", index=False, encoding="utf-8-sig")
latest_region_risk_ranking.to_csv(out_dir / "latest_region_risk_ranking.csv", index=False, encoding="utf-8-sig")
df.to_csv(out_dir / "modeling_result_dataset.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", out_dir)

저장 완료: ../outputs/tables
